In [1]:
import requests
import torch
import re
import time
import psutil
import subprocess
from unidecode import unidecode
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, hamming_loss

import pandas as pd

from datasets import load_dataset

In [2]:
ds = load_dataset("Rami/multi-label-class-github-issues-text-classification")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 778 entries, 0 to 777
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     778 non-null    object
 1   labels    778 non-null    object
 2   bodyText  778 non-null    object
dtypes: object(3)
memory usage: 18.4+ KB


In [3]:
labels = ["bug", "feature", "question", "won't fix", "docs"]

test = test[test['labels'].apply(lambda cats: all(c in labels for c in cats))]
test = test[test['labels'].apply(len) > 0]
test.rename(columns={'title': 'text'}, inplace=True)
test.drop(columns=['bodyText'], inplace=True)
test.reset_index(drop=True, inplace=True)

test

,text,labels
0,Update CONTRIBUTING.md on bugfixes/features PRs,[docs]
1,How to print the metric (across all working tr...,"[question, won't fix]"
2,Model loaded from checkpoint has bad accuracy,[question]
3,Add a robots.txt to stop Google from indexing ...,[docs]
4,Multi-processing with IterableDataset Warning,[docs]
...,...,...
195,TensorBoardLogger and ModelCheckpoint are not ...,[bug]
196,imagenet_example cannot run,[bug]
197,log_gpu_memory='all'` options raise Error,[bug]
198,`overfit_pct` vs `train_percent_check` etc,"[feature, docs]"


In [4]:
mlb = MultiLabelBinarizer()
test_labels_binarized = mlb.fit_transform(test['labels'])

test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

test = pd.concat([test, test_labels_df], axis=1)

test.drop(columns=['labels'], inplace=True)

In [5]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_19076\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


50761728

In [6]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [7]:
def classify(text, labels):
    url = "http://localhost:11434/api/chat"
    
    messages = [
        {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling multilabel classification tasks based on user instructions."},
        {"role": "user", "content": f"Classify the following text based on the task: Classification of github issues. Only respond with the labels that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
    ]
    
    start_time = time.time()

    try:
        response = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": True,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 3100
            }
        }, timeout=30)
        response_time = time.time() - start_time
        vram_usage = get_gpu_memory_usage()
        ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)
        response = response.json()
        response_text = response['message'].get('thinking', '') if 'message' in response else ''
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return "error", {}, 0, 0, 0, 0, f"API Error: {e}"
    
    if not response.get('done', False):
        print(f"Ollama returned an incomplete response: {response.get('error')}")
        return 'error', {}, response_time, 0, 0, 0, response.get('error', 'Incomplete response')
    
    if 'message' in response and 'content' in response['message']:
        classification_text = response['message']['content'].lower()
        print("Response fields:", ', '.join(response.keys()))
        print(response)
        total_time = response['total_duration'] / 1_000_000_000
    else:
        messages.append({"role": "assistant", "content": response_text + '</think>'})
        start_time2 = time.time()
        response2 = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": False,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 430
            }
        })
        response_time += time.time() - start_time2
        response2 = response2.json()
        classification_text = response2['message']['content'].lower() if 'message' in response2 and 'content' in response2['message'] else ''
        total_time = response['total_duration'] / 1_000_000_000 + response2['total_duration'] / 1_000_000_000
    
    label_counts = {label: len(re.findall(r'\b' + re.escape(label.lower()) + r'\b', classification_text)) for label in labels}

    list = []

    if 'bug' in classification_text:
        list.append('bug')
    if 'feature' in classification_text:
        list.append('feature')
    if 'question' in classification_text:
        list.append('question')
    if "won't fix" in classification_text:
        list.append("won't fix")
    if 'docs' in classification_text:
        list.append('docs')
    
    print(f"Text: {text}")
    print(f"Response: {list}")
    
    return list, label_counts, response_time, vram_usage, ram_usage_bytes, total_time, response_text

In [8]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'label_counts','response_time', 'vram_usage', 'ram_usage', 'total_time', 'response_text']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_19076\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


Response fields: model, created_at, message, done_reason, done, total_duration, load_duration, prompt_eval_count, prompt_eval_duration, eval_count, eval_duration
{'model': 'deepseek-r1:1.5b', 'created_at': '2025-06-15T17:35:04.8893285Z', 'message': {'role': 'assistant', 'content': 'The text describes the action of updating an existing contribution in the "bugfixes/features PRs" section of GitHub. This involves enhancing or improving an existing feature or contribution by modifying a specific file.\n\n**Labels:** feature', 'thinking': 'Okay, so I need to classify this text about updating a file called "CONTRIButing.md" in the "bugfixes/features PRs" section of GitHub. The task is to determine what kind of issue or question this update addresses. Let me break it down step by step.\n\nFirst, I\'ll look at the text: "Update CONTRIBUTING.md on bugfixes/features PRs." It seems like they\'re modifying a specific file related to bug reporting and feature requests. So, the main action here is u

In [9]:
for label in labels:
    test[f"{label} pred"] = test.apply(lambda row: 1 if label in row['prediction'] else 0, axis=1)

test = test.drop(columns=['prediction'])

test.to_csv('results/deepseekR1_ZS_multilabel2.csv', index=False)
test

,text,bug,docs,feature,question,won't fix,label_counts,response_time,vram_usage,ram_usage,total_time,response_text,bug pred,feature pred,question pred,won't fix pred,docs pred
0,Update CONTRIBUTING.md on bugfixes/features PRs,0,1,0,0,0,"{'bug': 0, 'feature': 2, 'question': 0, 'won't...",10.088812,2297,89.406250,8.050556,"Okay, so I need to classify this text about up...",1,1,0,0,0
1,How to print the metric (across all working tr...,0,0,0,1,1,"{'bug': 1, 'feature': 2, 'question': 2, 'won't...",4.301018,2297,89.500000,2.276702,"Okay, so I need to classify this GitHub issue ...",1,1,1,1,1
2,Model loaded from checkpoint has bad accuracy,0,0,0,1,0,"{'bug': 0, 'feature': 0, 'question': 0, 'won't...",3.736623,2297,89.660156,1.703974,"Okay, so I need to classify this text about a ...",0,0,0,0,0
3,Add a robots.txt to stop Google from indexing ...,0,1,0,0,0,"{'bug': 1, 'feature': 1, 'question': 0, 'won't...",4.680914,2288,89.777344,2.622718,"Okay, so I need to classify this text into one...",1,1,0,1,1
4,Multi-processing with IterableDataset Warning,0,1,0,0,0,"{'bug': 1, 'feature': 1, 'question': 1, 'won't...",6.674322,2288,89.808594,4.642125,"Okay, so I need to classify this text about Gi...",1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,TensorBoardLogger and ModelCheckpoint are not ...,1,0,0,0,0,"{'bug': 0, 'feature': 0, 'question': 0, 'won't...",7.925962,2303,77.902344,5.895883,"Okay, so I need to classify this text into one...",0,1,0,1,0
196,imagenet_example cannot run,1,0,0,0,0,"{'bug': 1, 'feature': 0, 'question': 0, 'won't...",7.215220,2303,78.457031,5.182121,"Okay, so I need to classify this text into one...",1,0,0,0,0
197,log_gpu_memory='all'` options raise Error,1,0,0,0,0,"{'bug': 1, 'feature': 1, 'question': 0, 'won't...",7.778123,2303,78.457031,5.724401,"Okay, so I'm trying to figure out how to class...",1,1,0,1,0
198,`overfit_pct` vs `train_percent_check` etc,0,1,1,0,0,"{'bug': 1, 'feature': 1, 'question': 1, 'won't...",7.003304,2303,78.156250,4.962235,"Okay, so I need to classify this text about Gi...",1,1,1,1,1


In [10]:
y_true = test[labels].values
y_pred = test[[f"{label} pred" for label in labels]].values

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)
hamming_loss = hamming_loss(y_true, y_pred)
print('Hamming loss: %f' % hamming_loss)

Accuracy: 0.135000
F1 score: 0.456466
Precision: 0.481553
Recall: 0.526316
Hamming loss: 0.450000


In [11]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 6.921880096197128
Average VRAM usage: 2301.855
Average RAM usage: 77.91107421875
Average total time: 4.8812359775


In [12]:
# save results to txt
with open('results/deepseekR1_ZS_multilabel2.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')
    f.write(f'Lines classified: {len(test)}\n')